# Phase 1: Business Understanding
## Documenting the Current HC Review Process

**CRISP-DM Purpose:** Define the business problem, objectives, and success criteria before any data work begins.

**Project:** ML-Driven FP&A Analyst -- Headcount Actuals vs. Forecast Analysis

**Important framing:** This notebook documents the **current state (as-is)** of the monthly headcount review process at Entrata. The purpose is to give the FP&A Analyst Agent a deep baseline understanding of how HC close works today -- every system, every handoff, every definition. **Automation and workflow redesign will come in a later phase.** No changes to the existing process are proposed here.

---

## 1. Current Process Map

The monthly headcount close follows this sequence, executed on the 1st (or 2nd) of each month after the prior month ends.

### Step-by-Step Process

| Step | Action | System | Owner |
|------|--------|--------|-------|
| 0 | **Wait for ADP snapshot.** The snapshot does not occur until 10:30 PM MT on the last day of the month. Data is not available until ~11 AM IST on the 1st. | ADP / Domo | Automated |
| 1 | **Verify Domo card is current.** The card "BETA \| FP&A \| HC Upload Writeback" pulls from ADP on the 1st of every month. If it hasn't auto-run, manually trigger via 3-dot menu > "Run Now". | Domo | India team |
| 2 | **Import into Adaptive Planning.** Go to Integration > Run Tasks > "HC - Upload". Confirm version settings: Actuals Version = "Actuals", CWM = "CWM", Current Month = prior month, Following Month = current month. Press Run. | Adaptive Planning | India team |
| 3 | **Handle dimension mapping failures.** If the import task fails, new dimensions likely need mapping. Get director-level input in #adaptive-gl-mapping channel. If failures persist, escalate to the Adaptive administrator. | Adaptive / Slack | India team |
| 4 | **Verify Existing Personnel.** Check the "Existing Personnel" sheet in Adaptive for both Actuals and CWM versions. | Adaptive Planning | India team |
| 5 | **Verify GAAP P/L accounts.** Check that actuals appear in: FTE - QBR, FTE - IC, R&D - FTE, New Hires (Actual New Hire [Entrata, RD, Colleen]), Attrition (Attrition [Entrata, RD, Colleen]). Confirm the following month's Attrition assumptions have been zeroed out. | Adaptive Planning | India team |
| 6 | **Leasing Center FTE upload (separate process).** Pull ADP "Actual vs. Scheduled Hours Report - Employee Summary" for last month. Paste Position ID and Actual Hours into the Google Sheet "Hours Report (Input)" tab. Process through "Combined (Output)" and update the LC Staffing sheet in Adaptive (CWM version). Manually enter FTE Existing and Distinct Headcount in GAAP P/L by Account > Leasing Center - US (Actuals version). | ADP / Google Sheets / Adaptive | India team |
| 7 | **FX rate check.** Rarely needed, but if directed: Adaptive > Design Integrations > Data Sources > 4a. Existing Personnel - Domo. Adjust INR, ILS, CAD, EUR rates in settings. | Adaptive Planning | India team |
| 8 | **Refresh the Close File.** Open OneDrive > FP&A > 08. Control Book > Close File - Master. Log into Office Connect, update Report Date in Workbook Properties (push one month out). Select all HC tabs and refresh via Office Connect > Refresh > Selected Sheets. | Close File (Excel + Office Connect) | FP&A team |
| 9 | **Post to Slack.** Screenshot each HC variance tab (Total, US, India, RD, Colleen) and post to #fpa_only with a templated message notifying the team that HC data has been uploaded. | Slack | FP&A team |
| 10 | **FP&A analysts tie out their areas.** Each analyst reviews their assigned departments against Domo, deletes actual starts from the new hire sheet, adds backfills, adjusts for planned terms that did not happen, and adds commentary to the HC tabs in the Close File. | Adaptive / Domo / Close File | FP&A analysts |
| 11 | **HC Review Meeting.** The team discusses variances between forecasted HC vs. actuals and updates the headcount forecast for the remainder of the year. | Meeting | Full FP&A team |

### Process Flow Diagram

```
ADP (employee system of record)
  |-- auto-snapshot last day of month -->
Domo ("BETA | FP&A | HC Upload Writeback" card)
  |-- joins with ADP-to-Adaptive Department Mapping -->
  |-- imports via Integration task -->
Adaptive Planning
  |-- Existing Personnel (Actuals + CWM versions)
  |-- New Hires / Terminations (Actuals + CWM versions)
  |-- Nominal HC (hard-coded from CWM into Actuals)
  |-- Leasing Center FTE (separate upload from ADP hours report)
  |
  |-- Office Connect refresh -->
Close File - Master (Excel)
  |-- HC tabs: Total, US, India, RD, Colleen
  |-- Screenshots -->
Slack (#fpa_only)
  |-- FP&A analysts tie out -->
HC Review Meeting
  |-- Variance discussion + forecast update
```

## 2. Stakeholder Map

The HC variance data is consumed by every level of the organization:

| Stakeholder | How They Use HC Variance Data |
|-------------|------------------------------|
| **FP&A Analysts** | Tie out assigned departments -- verify actuals match expectations, enter backfills for departures, remove actual starts from new hire sheet, provide commentary |
| **FP&A Leadership (VP Finance / CFO)** | Review at the HC Review Meeting -- assess overall HC trajectory, approve forecast updates |
| **Department Owners / VPs** | Held accountable for HC in their area -- explain variances, justify open headcount |
| **Board / Executive Team** | Rolled-up HC data feeds into board-level reporting and company-wide workforce metrics |
| **HR / People & Places** | Workforce planning alignment -- ensure hiring pipeline matches approved headcount plan |

### Key Decision Points Driven by HC Variance Data

- **Hiring cleanse:** Each month, planned hires that have already started are removed and backfills are added for departures
- **Forecast update:** The CWM is updated based on actual HC trajectory -- if a department is behind on hiring, the remaining-year forecast may be adjusted
- **Attrition handling:** If a planned termination did not occur, the term date is pushed to the following month
- **Budget accountability:** Persistent over/under-forecast by department triggers conversations with department owners

## 3. Department Taxonomy

The Close File organizes headcount into a 3-level hierarchy. The taxonomy below is extracted directly from the HC (Total) tab of the Close File - Master.

### Hierarchy Structure

```
Entrata (Total Company)
|
+-- COR (Cost of Revenue)
|   +-- COR Only *
|   +-- Fulfillment
|   |   +-- ResidentUtility
|   |   +-- UEM
|   |   +-- Invoice Processing
|   +-- Implementation
|   +-- Leasing Center
|   +-- ResidentInsure
|   +-- Site Reliability
|   +-- Support
|   +-- Training
|   +-- Digital Marketing
|   +-- ResidentVerify
|
+-- S&M (Sales & Marketing)
|   +-- Sales
|   +-- Marketing
|   +-- Customer Success
|
+-- R&D (Research & Development)
|   +-- Engineering
|   +-- Product
|
+-- G&A (General & Administrative)
    +-- Finance
    +-- Executive
    +-- Legal
    +-- People & Places
    +-- IT
    +-- SBO
    +-- Payments
```

\* **COR Only should never have headcount.** All COR Only rows (including sub-entity splits) should always be 0.

### Entity Sub-Splits

Each department is further broken down by entity. Each employee maps to exactly **one** entity:

- **US** -- United States operations
- **India** -- India operations
- **RD** -- Paris / R&D entity
- **Colleen** -- Colleen subsidiary

**Total should always equal US + India + RD + Colleen.** Small rounding differences may occur due to Leasing Center FTE calculations (hours-based).

### Summary Row Structure (from Close File)

Each HC tab also contains these summary rows at the bottom:

| Row | Description |
|-----|-------------|
| Total OPEX | S&M + R&D + G&A headcount (all operating expense departments) |
| Total Entrata | All headcount (= COR + Total OPEX) |
| HC excl. LC | Total Entrata minus Leasing Center FTE |

### Departments with Notable Entity Concentrations (Mar-2025 snapshot)

| Department | Total | US | India | RD | Colleen |
|-----------|-------|-----|-------|-----|----------|
| Engineering | 667 | 89 | 542 | 16 | 20 |
| Leasing Center | 167.8 | 167.8 | 0 | 0 | 0 |
| Implementation | 170 | 100 | 65 | 3 | 2 |
| Support | 127 | 98 | 4 | 25 | 0 |
| Sales | 116 | 113 | 0 | 2 | 1 |
| Product | 101 | 67 | 28 | 4 | 2 |
| **Total Entrata** | **2001.8** | **1039.8** | **849** | **81** | **32** |

Note: Engineering is overwhelmingly India-based (81%). Leasing Center is 100% US. These concentrations matter for variance analysis -- a single hire/departure in a small-entity department creates a larger % variance.

## 4. Data Source Inventory

### Systems and Their Roles

| System | Role in HC Process | Data It Holds |
|--------|-------------------|---------------|
| **ADP** | Employee system of record | Employee census, hours worked, position IDs, hire/term dates |
| **Domo** | Data integration layer | HC Upload Writeback card (ADP snapshot + department mapping), HC validation cards |
| **Adaptive Planning** | Forecasting and actuals repository | Existing Personnel, New Hires, Terminations, Nominal HC across Actuals, CWM, and Latest Forecast versions |
| **Close File - Master (Excel)** | Reporting and variance analysis | HC tabs refreshed via Office Connect from Adaptive, with variance calculations |
| **Google Sheets** | Leasing Center FTE calculation | Hours Report input, Census input, Combined output, Pivot Table for LC staffing |
| **Slack (#fpa_only)** | Communication and distribution | HC variance screenshots, tie-out instructions, HC Review meeting scheduling |

### Data Flow Diagram

```
ADP ──snapshot──> Domo Card ──import──> Adaptive Planning ──Office Connect──> Close File
                    |                        |                                    |
                    |                        +-- Actuals version                  +-- HC (Total)
                    |                        +-- CWM version                      +-- HC-US
                    |                        +-- Latest Forecast version           +-- HC-India
                    |                                                              +-- HC-RD
ADP Hours Report ──> Google Sheets ──manual──> Adaptive (LC FTE)                  +-- HC-Colleen
```

### Adaptive Planning Versions (Critical for Variance Logic)

| Version | What It Contains |
|---------|------------------|
| **Actuals** | The loaded actual headcount from ADP for completed months |
| **CWM (Current Working Model)** | Rolling forecast -- contains actuals for past months plus the latest forecast for future months |
| **Latest Forecast** | The prior month's approved forecast -- this is the comparison baseline for variance analysis |

## 5. Key Data Definitions

These definitions were confirmed with the FP&A Manager and are the authoritative source for all downstream work in this project.

### Headcount Metric: FTE

- **For all departments except Leasing Center:** Every employee (full-time or part-time) counts as **1 distinct headcount**. FTE = distinct headcount.
- **For Leasing Center:** FTE is calculated based on **hours worked**, derived from the ADP "Actual vs. Scheduled Hours Report." This is why Leasing Center shows decimal values (e.g., 167.8 FTE).
- **Contractors and temps** are **not** included in the HC count.

### Entity Allocation

- Each employee is mapped to **exactly one legal entity**: US, India, RD (Paris), or Colleen.
- **Total = US + India + RD + Colleen** (always). Small rounding differences may exist due to Leasing Center FTE calculations.

### Variance Calculations

**Month Variance:**
- Formula: **Actuals - Latest Forecast**
- Sign convention: **Positive = over forecast** (more heads than planned), **Negative = under forecast** (fewer heads than planned)
- Comparison: CWM for the prior month (which now contains actuals) vs. the Latest Forecast for that same prior month (what we forecasted last month)

**EOY Forecast Variance:**
- Formula: **CWM full-year - Latest Forecast full-year**
- Compares the current full-year CWM forecast against the Latest Forecast full-year projection

### COR Only

- **COR Only should never have headcount.** All COR Only rows across all entity tabs should always show 0. If they don't, it is a data issue to be flagged.

## 6. Close File HC Tab Schema

Each of the 5 HC tabs (HC, HC-US, HC-India, HC-RD, HC-Colleen) follows the same layout.

### Column Structure

| Column(s) | Content |
|-----------|--------|
| B | Department name (row labels) |
| C through M | Monthly actuals (rolling ~12 months, e.g., Mar-2025 through Jan-2026) |
| N | Latest actuals month (used for Month Variance) |
| P | Actuals for the variance comparison month |
| Q | Latest Forecast for the same month |
| V | CWM (EOY Forecast Variance reference) |

### Row Structure

- **Row 4:** Column headers (month labels)
- **Row 6:** Entrata (total company)
- **Rows 7-164:** Department breakdown with entity sub-splits
- **Row 166:** Total OPEX
- **Row 167:** Total Entrata
- **Row 169:** HC excl. LC

### Variance Sections

- **Column N area:** Month Variance (Actuals vs. Latest Forecast for the most recent closed month)
- **Column V area:** EOY Forecast Variance (CWM full-year vs. Latest Forecast full-year)

## 7. Success Criteria for This Project

### Phase 1 (This Notebook) -- COMPLETE when:
- The FP&A Analyst Agent has a documented, confirmed understanding of the current HC process
- All data definitions, department taxonomy, and variance logic are locked down with manager sign-off
- No new workflows or automation are proposed -- this is purely current-state documentation

### Future Phases (to be defined collaboratively):
- **Phase 2 (Data Understanding):** Profile the actual data in the Close File and upstream systems
- **Phase 3 (Data Preparation):** Build the data pipeline to extract and join actuals vs. forecast
- **Phase 4 (Analysis & Modeling):** Layered approach:
  - Layer 1: Automate the monthly HC variance report (workflow to be redesigned with the FP&A Analyst Agent)
  - Layer 2: Deeper analytics -- trend analysis, variance pattern detection, hire/attrition decomposition
  - Layer 3: Predictive modeling -- forecast future HC trajectory and flag likely deviations
- **Phase 5 (Evaluation):** Validate outputs, assess accuracy, produce executive summary

**The automation layer (Layer 1) will be a collaborative redesign** -- not a replication of the current manual process. The agent and the FP&A Manager will rethink how the HC review workflow should work with an AI-assisted approach.

## 8. Decisions Log

All decisions confirmed with the FP&A Manager during Phase 1 are recorded here for audit trail.

| # | Decision | Confirmed By | Date |
|---|----------|-------------|------|
| 1 | Process map (11 steps) is accurate and complete | FP&A Manager | 2026-04-01 |
| 2 | India team handles the upload; rest of FP&A team handles variance review | FP&A Manager | 2026-04-01 |
| 3 | All stakeholder groups consume HC variance data (FP&A, leadership, dept owners, board, HR) | FP&A Manager | 2026-04-01 |
| 4 | HC Review Meeting covers both variance discussion AND forecast updates | FP&A Manager | 2026-04-01 |
| 5 | COR Only should never have headcount | FP&A Manager | 2026-04-01 |
| 6 | Sub-department entity breakdowns (US/India/RD/Colleen) matter for the analysis | FP&A Manager | 2026-04-01 |
| 7 | FTE definition: all employees = 1 HC (FT and PT), except Leasing Center (hours-based) | FP&A Manager | 2026-04-01 |
| 8 | Month Variance = Actuals - Latest Forecast (positive = over forecast) | FP&A Manager | 2026-04-01 |
| 9 | Forecast comparison: Latest Forecast for prior month vs. CWM for prior month | FP&A Manager | 2026-04-01 |
| 10 | Entity allocation: each employee maps to one entity; Total = sum of 4 entities | FP&A Manager | 2026-04-01 |
| 11 | Leasing Center FTE rounding can cause small Total vs. sum-of-entities differences | FP&A Manager | 2026-04-01 |
| 12 | EOY Forecast Variance = CWM full-year vs. Latest Forecast full-year | FP&A Manager | 2026-04-01 |
| 13 | Automation will be a collaborative redesign, not a replication of the manual process | FP&A Manager | 2026-04-01 |

---

**Phase 1 Status: COMPLETE**

The FP&A Analyst Agent now has a documented understanding of the current HC review process, all data definitions, the department taxonomy, and the system landscape. This serves as the foundation for Phase 2 (Data Understanding), where we will profile the actual data and identify quality issues.

**Next step:** Phase 2 -- Data Understanding (to be initiated upon manager approval)